## Stock Price Prediction with LSTM
Using LSTM is one of the best machine learning approaches for time series forecasting. LSTMs are recurrent neural networks designed to remember data for a longer period. So, whenever you are working on a problem where your neural network fails to memorize data, you can use LSTM neural network.

In [7]:
import pandas as pd
import yfinance as yf
import datetime
from datetime import date, timedelta
today = date.today()

d1 = today.strftime("%Y-%m-%d")
end_date = d1
d2 = date.today() - timedelta(days=5000)
d2 = d2.strftime("%Y-%m-%d")
start_date = d2

data = yf.download('AAPL',
                      start=start_date,
                      end=end_date,
                      progress=False)

# Flatten multi-level columns if they exist
if isinstance(data.columns, pd.MultiIndex):
    data.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in data.columns.values]
    # Clean up column names to remove 'AAPL' ticker if it was added
    data.columns = data.columns.str.replace('_AAPL', '', regex=False)

data["Date"] = data.index
data = data[["Date", "Open", "High", "Low", "Close",
             "Volume"]]
data.reset_index(drop=True, inplace=True)
data.tail()

/tmp/ipython-input-1340268514.py:13: FutureWarning:

YF.download() has changed argument auto_adjust default to True



,Date,Open,High,Low,Close,Volume
3437,2025-12-15,280.149994,280.149994,272.839996,274.109985,50409100
3438,2025-12-16,272.820007,275.500000,271.790009,274.609985,37648600
3439,2025-12-17,275.010010,276.160004,271.640015,271.839996,50138700
3440,2025-12-18,273.609985,273.630005,266.950012,272.190002,51630700
3441,2025-12-19,272.149994,274.600006,269.899994,273.670013,144599200


In [8]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data["Date"],
                                        open=data["Open"],
                                        high=data["High"],
                                        low=data["Low"],
                                        close=data["Close"])])
figure.update_layout(title = "Apple Stock Price Analysis",
                     xaxis_rangeslider_visible=False)
figure

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3442 entries, 0 to 3441
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    3442 non-null   datetime64[ns]
 1   Open    3442 non-null   float64       
 2   High    3442 non-null   float64       
 3   Low     3442 non-null   float64       
 4   Close   3442 non-null   float64       
 5   Volume  3442 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 161.5 KB


In [10]:
correlation = data.corr()
print(correlation["Close"].sort_values(ascending=False))

Close     1.000000
High      0.999888
Low       0.999885
Open      0.999753
Date      0.932127
Volume   -0.538098
Name: Close, dtype: float64


Training LSTM for Stock Price Prediction

In [11]:
features = data[["Open", "High", "Low", "Close", "Volume"]]



In [12]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(features)


In [13]:
import numpy as np

def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size, 3])  # Close (index=3)
    return np.array(X), np.array(y)


In [14]:
WINDOW_SIZE = 10
X, y = create_sequences(scaled_data, WINDOW_SIZE)


In [16]:
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]


In [17]:
from keras.models import Sequential
from keras.layers import LSTM, Dense

model = Sequential()
model.add(LSTM(64, return_sequences=True,
               input_shape=(WINDOW_SIZE, X.shape[2])))
model.add(LSTM(32))
model.add(Dense(1))

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 10, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,369 (118.63 KB)

 Trainable params: 30,369 (118.63 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0140 - val_loss: 0.0014
Epoch 2/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.1855e-04 - val_loss: 0.0019
Epoch 3/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.2304e-04 - val_loss: 0.0016
Epoch 4/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 1.1956e-04 - val_loss: 0.0020
Epoch 5/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.1617e-04 - val_loss: 0.0020
Epoch 6/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 1.0708e-04 - val_loss: 0.0027
Epoch 7/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 1.0925e-04 - val_loss: 0.0024
Epoch 8/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.2606e-04 - val_loss: 0.0029
Epoch 9/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.1042e-04 - val_loss: 0.0012
Epoch 10/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.3309e-04 - val_loss: 0.0019
Epoch 11/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 9.4174e-05 - val_loss: 0.0018
Epoch 12/30
86/86 ━━━━━

In [19]:
predicted = model.predict(X_test)


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step


In [20]:
close_scaler = MinMaxScaler()
close_scaler.min_, close_scaler.scale_ = scaler.min_[3], scaler.scale_[3]

predicted = close_scaler.inverse_transform(predicted.reshape(-1,1))
y_test_real = close_scaler.inverse_transform(y_test.reshape(-1,1))


In [21]:
last_window = scaled_data[-WINDOW_SIZE:]
last_window = last_window.reshape(1, WINDOW_SIZE, scaled_data.shape[1])

next_day_prediction = model.predict(last_window)
next_day_price = close_scaler.inverse_transform(next_day_prediction)

print("Yarınki tahmini Close fiyatı:", next_day_price[0][0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Yarınki tahmini Close fiyatı: 259.9866


In [22]:
import joblib

joblib.dump(scaler, "feature_scaler.joblib")
joblib.dump(close_scaler, "close_scaler.joblib")


['close_scaler.joblib']